# Generative networks

**AI3 · NLP Deep Dive** — [AI Olympiad Curriculum](https://abzlitochka.github.io/ai-olympiad-curriculum)

~15 min

## Introduction

Recurrent Neural Networks (RNNs) and their gated cell variants such as Long Short Term Memory Cells (LSTMs) and Gated Recurrent Units (GRUs) provided a mechanism for language modeling in that they can learn word ordering and provide predictions for the next word in a sequence. This allows us to use RNNs for **generative tasks**, such as ordinary text generation, machine translation, and even image captioning.

> ✅ Think about all the times you've benefited from generative tasks such as text completion as you type. Do some research into your favorite applications to see if they leveraged RNNs.

In RNN architecture we discussed in the previous unit, each RNN unit produced the next hidden state as an output. However, we can also add another output to each recurrent unit, which would allow us to output a **sequence** (which is equal in length to the original sequence). Moreover, we can use RNN units that do not accept an input at each step, and just take some initial state vector, and then produce a sequence of outputs.

This allows for different neural architectures that are shown in the picture below:

![Image showing common recurrent neural network patterns.](https://raw.githubusercontent.com/microsoft/AI-For-Beginners/main/lessons/5-NLP/17-GenerativeNetworks/images/unreasonable-effectiveness-of-rnn.jpg)

> Image from blog post [Unreasonable Effectiveness of Recurrent Neural Networks](http://karpathy.github.io/2015/05/21/rnn-effectiveness/) by [Andrej Karpaty](http://karpathy.github.io/)

* **One-to-one** is a traditional neural network with one input and one output
* **One-to-many** is a generative architecture that accepts one input value, and generates a sequence of output values. For example, if we want to train an **image captioning** network that would produce a textual description of a picture, we can a picture as input, pass it through a CNN to obtain its hidden state, and then have a recurrent chain generate caption word-by-word
* **Many-to-one** corresponds to the RNN architectures we described in the previous unit, such as text classification
* **Many-to-many**, or **sequence-to-sequence** corresponds to tasks such as **machine translation**, where we have first RNN collect all information from the input sequence into the hidden state, and another RNN chain unrolls this state into the output sequence.

In this unit, we will focus on simple generative models that help us generate text. For simplicity, we will use character-level tokenization.

We will train this RNN to generate text step by step. On each step, we will take a sequence of characters of length `nchars`, and ask the network to generate the next output character for each input character:

![Image showing an example RNN generation of the word 'HELLO'.](ms-assets/en-originals/rnn-generate.png)

When generating text (during inference), we start with some **prompt**, which is passed through RNN cells to generate its intermediate state, and then from this state the generation starts. We generate one character at a time, and pass the state and the generated character to another RNN cell to generate the next one, until we generate enough characters.

![Illustration](https://raw.githubusercontent.com/microsoft/AI-For-Beginners/main/lessons/5-NLP/17-GenerativeNetworks/images/rnn-generate-inf.png)

> Image by the author



## Exercises: Generative Networks

Continue your learning in the following notebooks:

* [Generative Networks with PyTorch](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/5-NLP/17-GenerativeNetworks/GenerativePyTorch.ipynb)
* [Generative Networks with TensorFlow](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/5-NLP/17-GenerativeNetworks/GenerativeTF.ipynb)



## Soft text generation and temperature

The output of each RNN cell is a probability distribution of characters. If we always take the character with the highest probability as the next character in generated text, the text often can become "cycled" between the same character sequences again and again, like in this example:



In [ ]:
today of the second the company and a second the company ...

However, if we look at the probability distribution for the next character, it could be that the difference between a few highest probabilities is not huge, e.g. one character can have probability 0.2, another - 0.19, etc. For example, when looking for the next character in the sequence '*play*', next character can equally well be either space, or **e** (as in the word *player*).

This leads us to the conclusion that it is not always "fair" to select the character with a higher probability, because choosing the second highest might still lead us to meaningful text. It is more wise to **sample** characters from the probability distribution given by the network output. We can also use a parameter, **temperature**, that will flatten out the probability distribution, in case we want to add more randomness, or make it more steep, if we want to stick more to the highest-probability characters.

Explore how this soft text generation is implemented in the notebooks linked above.



## Conclusion

While text generation may be useful in its own right, the major benefits come from the ability to generate text using RNNs from some initial feature vector. For example, text generation is used as part of machine translation (sequence-to-sequence, in this case state vector from *encoder* is used to generate or *decode* translated message), or generating textual description of an image (in which case the feature vector would come from CNN extractor).



## Challenge

Take some lessons on Microsoft Learn on this topic

* Text Generation with [PyTorch](https://learn.microsoft.com/learn/modules/intro-natural-language-processing-pytorch/6-generative-networks/?WT.mc_id=academic-77998-cacaste)/[TensorFlow](https://learn.microsoft.com/learn/modules/intro-natural-language-processing-tensorflow/5-generative-networks/?WT.mc_id=academic-77998-cacaste)



## Review & Self Study

Here are some articles to expand your knowledge

* Different approaches to text generation with Markov Chain, LSTM and GPT-2: [blog post](https://towardsdatascience.com/text-generation-gpt-2-lstm-markov-chain-9ea371820e1e)
* Text generation sample in [Keras documentation](https://keras.io/examples/generative/lstm_character_level_text_generation/)



## Assignment

We have seen how to generate text character-by-character. In the lab, you will explore word-level text generation.



---

## ✅ Self-check tasks

Implement the function in the 'your code' cell, then run the check cell — it runs randomized tests and shows the first mismatch. Run the helper cell below once first.

In [ ]:
# ── task-checking helper cell — run once ──
import copy
import numpy as np
import math, statistics, itertools, collections, functools, heapq, random

try:
    import sys; sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

_L = {"noFn":"❓ Define a function `{e}` in the solution cell and run it","runErr":"❌ Test {i}: {t}: {m}","wrong":"❌ Test {i}: {r}","badRef":"⚠️ Check harness broke (generator/reference): {t}: {m}","inL":"Input","expL":"Expected","gotL":"Got","ok":"✅ Correct! All {n} tests passed."}

def _nb_copy_kw(kw):
    return {k: (v.copy() if isinstance(v, np.ndarray) else copy.deepcopy(v)) for k, v in kw.items()}

def _nb_jsafe(x, depth=0):
    if depth > 6: return "…"
    if isinstance(x, np.ndarray): return _nb_jsafe(x.tolist(), depth + 1)
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.floating): return float(x)
    if isinstance(x, np.bool_): return bool(x)
    if isinstance(x, float): return round(x, 6) if math.isfinite(x) else str(x)
    if isinstance(x, (list, tuple)): return [_nb_jsafe(v, depth + 1) for v in x]
    if isinstance(x, dict): return {str(k): _nb_jsafe(v, depth + 1) for k, v in x.items()}
    if isinstance(x, (int, str, bool)) or x is None: return x
    return repr(x)

def _nb_cmp(exp, got, path=""):
    if isinstance(exp, np.ndarray) or isinstance(got, np.ndarray):
        e, g = np.asarray(exp), np.asarray(got)
        if e.shape != g.shape:
            return f"shape {tuple(g.shape)} != expected {tuple(e.shape)}"
        try:
            if np.issubdtype(e.dtype, np.number) or np.issubdtype(g.dtype, np.number):
                ef, gf = e.astype(float), g.astype(float)
                close = np.isclose(gf, ef, atol=1e-6, rtol=1e-5, equal_nan=True)
                if not bool(close.all()):
                    i = tuple(int(v) for v in np.argwhere(~close)[0])
                    return f"{path}first mismatch at index {i}: got {gf[i]:.6g} vs expected {ef[i]:.6g}"
            elif not np.array_equal(g, e):
                return f"{path}values differ"
        except Exception as ex:
            return f"{path}cannot compare: {ex}"
        return None
    if isinstance(exp, (int, float)) and not isinstance(exp, bool) and isinstance(got, (int, float)) and not isinstance(got, bool):
        if not np.isclose(float(exp), float(got), atol=1e-6, rtol=1e-5):
            return f"{path}{got} != expected {exp}"
        return None
    if isinstance(exp, (list, tuple)) and isinstance(got, (list, tuple)):
        if len(exp) != len(got):
            return f"{path}length {len(got)} != expected {len(exp)}"
        for i, (a, b) in enumerate(zip(exp, got)):
            r = _nb_cmp(a, b, path + f"[{i}].")
            if r: return r
        return None
    if isinstance(exp, dict) and isinstance(got, dict):
        if set(exp) != set(got):
            return f"{path}keys differ"
        for k in exp:
            r = _nb_cmp(exp[k], got[k], path + f"[{k!r}].")
            if r: return r
        return None
    if exp != got:
        return f"{path}{got!r} != expected {exp!r}"
    return None

def _check_task(g, ref_src, entry, n=20, seed=0):
    """g — globals() ноутбука; ref_src — gen() + эталонное решение."""
    fn = g.get(entry)
    if not callable(fn):
        print(_L["noFn"].format(e=entry)); return
    ns = {"np": np, "math": math, "statistics": statistics, "itertools": itertools,
          "collections": collections, "functools": functools, "heapq": heapq, "random": random}
    try:
        exec(ref_src, ns)
        gen, ref = ns["gen"], ns[entry]
    except Exception as e:
        print(_L["badRef"].format(t=type(e).__name__, m=e)); return
    rng = np.random.default_rng(seed)
    for i in range(n):
        try:
            kw = gen(rng)
        except Exception as e:
            print(_L["badRef"].format(t=type(e).__name__, m=e)); return
        try:
            got = fn(**_nb_copy_kw(kw))
        except Exception as e:
            print(_L["runErr"].format(i=i + 1, t=type(e).__name__, m=e)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))); return
        exp = ref(**_nb_copy_kw(kw))
        r = _nb_cmp(exp, got)
        if r:
            print(_L["wrong"].format(i=i + 1, r=r)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))
                  + "\n" + _L["expL"] + ": " + repr(_nb_jsafe(exp))
                  + "\n" + _L["gotL"] + ": " + repr(_nb_jsafe(got))); return
    print(_L["ok"].format(n=n))

### Task 1 — Softmax with temperature

●○○ · ~8 min

> LLM decoding starts from logits and turns them into probabilities with softmax. **Temperature** T divides the logits first: softmax(z / T). T < 1 sharpens the distribution (more confident), T > 1 flattens it (more random). Always subtract the max before exp() for numerical stability.

Implement `softmax_temp(logits, T)` → the probability vector softmax(logits / T). It must sum to 1.

<details><summary>Hint</summary>

z = logits / T; subtract z.max(); exponentiate; divide by the sum.

</details>


In [ ]:
import numpy as np

def softmax_temp(logits, T):
    logits = np.asarray(logits, dtype=float)
    # your code — divide by T, then a numerically-stable softmax
    return logits / T  # wrong — no softmax

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    n = int(rng.integers(2, 15))\n    logits = rng.normal(0, 3, n)\n    if rng.random() < 0.4:\n        # logits far beyond exp(709): a naive softmax without max-subtraction overflows\n        logits = logits + float(rng.uniform(4000, 5000))\n    return {\"logits\": logits,\n            \"T\": float(rng.choice([0.3, 0.5, 0.7, 1.0, 2.0, 5.0]))}\n\ndef softmax_temp(logits, T):\n    z = np.asarray(logits, dtype=float) / T\n    z = z - z.max()\n    e = np.exp(z)\n    return e / e.sum()", "softmax_temp", 20)

### Task 2 — Top-k sampling filter

●●○ · ~10 min

> Top-k sampling keeps only the k highest-probability tokens, renormalizes, and samples — preventing the model from picking very unlikely tokens. The random part needs an RNG, but the *candidate set* is deterministic: it's just the k largest probabilities.

Implement `topk_ids(probs, k)` → the list of token indices allowed by top-k sampling, ordered by **decreasing** probability (all probabilities are distinct).

<details><summary>Hint</summary>

np.argsort sorts ascending — negate the probabilities, then take the first k and .tolist().

</details>


In [ ]:
import numpy as np

def topk_ids(probs, k):
    # your code
    return list(range(k))

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    n = int(rng.integers(5, 25))\n    p = rng.random(n) + 0.01\n    return {\"probs\": p / p.sum(), \"k\": int(rng.integers(1, n + 1))}   # k == n is legal — exercises the full-sort path\n\ndef topk_ids(probs, k):\n    p = np.asarray(probs, dtype=float)\n    return np.argsort(-p)[:k].tolist()", "topk_ids", 20)

### Task 3 — One beam-search expansion step

●●○ · ~12 min

> Beam search decodes text by keeping the `k` best partial hypotheses (a 'beam' is one partial output with a running score; each score is a sum of per-token log-probabilities, so longer outputs aren't unfairly favored until you normalize). One step expands every beam by every vocab token: candidate score = cumulative beam score + token log-prob. Sorting all B·V candidates and keeping the top k gives the next beam — deterministic, no sampling. A consistent tie-break (lower beam index, then lower token index) keeps results reproducible.

Implement `beam_step(beam_scores, logprobs, k)` → list of `(beam_idx, token_idx, new_score)` tuples for the `k` best expansions, where `new_score = beam_scores[i] + logprobs[i, j]`. Sort by **decreasing** score; break ties by smaller `beam_idx`, then smaller `token_idx`. Indices must be plain ints, scores plain floats.

```
beam_scores=[-1,-1], logprobs=[[-0.5,-0.5],[-0.2,-0.9]], k=2
→ scores: (1,0)=-1.2, (0,0)&(0,1) tie at -1.5 (lower beam_idx wins) → [(1,0,-1.2),(0,0,-1.5)]
```

<details><summary>Hint</summary>

Loop i over beams, j over vocab; collect (i, j, beam[i] + lp[i, j]); sort with key=lambda t: (-t[2], t[0], t[1]); take the first k.

</details>


In [ ]:
import numpy as np

def beam_step(beam_scores, logprobs, k):
    bs = np.asarray(beam_scores, dtype=float)
    lp = np.asarray(logprobs, dtype=float)
    # your code
    return []

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    B = int(rng.integers(2, 5)); V = int(rng.integers(3, 9))\n    # rounded to 1 decimal → candidate sums collide, so the tie-break rule is exercised\n    return {\"beam_scores\": np.round(-rng.uniform(0, 5, B), 1),\n            \"logprobs\": np.round(-rng.uniform(0, 4, (B, V)), 1),\n            \"k\": int(rng.integers(1, B * V + 1))}\n\ndef beam_step(beam_scores, logprobs, k):\n    bs = np.asarray(beam_scores, dtype=float)\n    lp = np.asarray(logprobs, dtype=float)\n    cand = []\n    for i in range(bs.shape[0]):\n        for j in range(lp.shape[1]):\n            cand.append((int(i), int(j), float(bs[i] + lp[i, j])))\n    cand.sort(key=lambda t: (-t[2], t[0], t[1]))\n    return cand[:k]", "beam_step", 20)

---

**Go deeper**
- [Original lesson (Microsoft, MIT)](https://github.com/microsoft/AI-For-Beginners/blob/main/lessons/5-NLP/17-GenerativeNetworks)
- [Yandex ML handbook (RU): generative models](https://contest.yandex.ru/tracks/ml/generative-models/introduction-to-generative-modeling)